# 03a — Shared Survival-Modeling Dataset (`v1`)

One narrow, versioned modeling dataset that Person 2 (feature-combination analysis), Person 3
(alternative survival models), and Person 4 (centralized evaluation + NACC harmonization) can all
load — so nobody re-joins raw ADNI data.

## Built ONLY from the frozen primary cohort
Input: `outputs/01c_mci_survival_cohort_freeze/mci_survival_primary_cohort_v1.tsv` (Build 1, v1),
SHA-256 `a15e2cb8…` — verified before anything else runs; the build refuses to continue on a
mismatch.

This notebook **does not**: re-join raw tables · re-anchor · redefine MCI eligibility · recompute
event/censor dates · exclude short-follow-up or prior-dementia participants · impute · scale ·
winsorize · select features · add cognitive or functional variables · fit models · generate CV folds.

## Design
- **All 401 participants preserved.** Optional biomarkers stay **NA where missing** — the dataset is
  NOT reduced to any complete-case subset.
- Raw biomarker values carried byte-for-byte **and retained for auditability**; the only derived
  variables are `APOE4_CARRIER`, `log_ptau217`, `log_gfap`, `log_nfl` (natural logs via
  `numpy.log`), the already-frozen `ratio_ab42_ab40`, and 8 per-participant `eligible_*`
  analysis-set flags.
- Two prior-dementia QC fields are carried so downstream sensitivity analyses need no re-join. They
  are **never predictors and never a filter** for this dataset.

**Exactly 26 columns, in a fixed order (asserted).**

## Deterministic by construction
No runtime timestamps, no randomness, no environment-dependent content in any generated artifact.
The dataset, data dictionary, analysis-set counts, manifest and build report all reproduce
byte-identically from a clean kernel. `20260722` is reserved for **downstream** cross-validation
only — no folds are generated here.

## Sections
0. Setup, read-only guards, input checksum · 1. Load and verify the frozen cohort ·
2. Assemble the 26 columns · 3. Validation · **3b. Missingness structure + prior-dementia** ·
4. Analysis-set counts · 5. Data dictionary · 6. Outputs + manifest · 7. Summary

## 0. Setup, read-only guards, and input checksum verification

In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
import subprocess
import sys
from pathlib import Path

# NOTE: datetime is deliberately NOT imported. This build writes no runtime timestamp into any
# artifact, and consumes no randomness — every generated file must be byte-identical across
# clean-kernel executions.

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)


def find_project_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "Data" / "raw").is_dir():
            return cand
    raise FileNotFoundError("Could not locate Data/raw above %s" % start)


PROJECT_ROOT = find_project_root(Path.cwd())
FREEZE_DIR = PROJECT_ROOT / "outputs" / "01c_mci_survival_cohort_freeze"
OUT_DIR = PROJECT_ROOT / "outputs" / "03a_mci_survival_shared_modeling"
OUT_DIR.mkdir(parents=True, exist_ok=True)

VERSION = "v1"
INPUT_NAME = "mci_survival_primary_cohort_v1.tsv"
INPUT_PATH = FREEZE_DIR / INPUT_NAME
EXPECTED_SHA = "a15e2cb8606f676b3bcfc4d23de72c752779836ce9406b04b01c84394a76e8ad"
EXPECTED = dict(n=401, events=85, censored=316, cols=78)

# Reserved for DOWNSTREAM cross-validation only. This notebook generates no folds and uses no RNG.
RESERVED_DOWNSTREAM_CV_SEED = 20260722

# ---- THE fixed 26-column output contract, in order ----
FINAL_COLUMNS = [
    "RID", "anchor_date",
    "time_to_event_or_censor_days", "event_indicator",
    "entry_age", "APOE4_COUNT", "APOE4_CARRIER", "ptau217", "log_ptau217",
    "gfap", "log_gfap", "nfl", "log_nfl", "abeta42", "abeta40", "ratio_ab42_ab40",
    "pre_anchor_dementia_n", "qc_dementia_on_or_before_anchor",
    "eligible_core", "eligible_gfap", "eligible_nfl", "eligible_amyloid",
    "eligible_gfap_nfl", "eligible_gfap_amyloid", "eligible_nfl_amyloid", "eligible_full_blood",
]

# Every existing frozen v1 artifact is READ-ONLY. Hashed before/after; asserted unchanged.
PROTECTED = [
    "mci_survival_anchor_cohort_v1.tsv", "mci_survival_followup_cohort_v1.tsv",
    "mci_survival_primary_cohort_v1.tsv", "mci_survival_exclusion_log_v1.tsv",
    "mci_survival_cohort_flow_v1.tsv", "mci_survival_freeze_provenance_v1.json",
    "mci_survival_manual_qc_cases_v1.tsv", "mci_survival_manual_qc_longitudinal_context_v1.tsv",
    "mci_survival_manual_qc_form_v1.tsv", "mci_survival_manual_qc_sampling_summary_v1.tsv",
    "mci_survival_feature_missingness_v1.tsv", "mci_survival_model_complete_case_counts_v1.tsv",
    "mci_survival_feature_timing_v1.tsv", "mci_survival_feature_scenario_counts_v1.tsv",
    "mci_survival_ptau181_platform_availability_v1.tsv", "mci_survival_data_dictionary_v1.tsv",
    "mci_survival_freeze_manifest_v1.json",
]

ASSERTIONS: list[dict] = []


def check(name, ok, detail=""):
    ok = bool(ok)
    ASSERTIONS.append({"assertion": name, "passed": ok, "detail": str(detail)})
    if not ok:
        raise AssertionError(f"ASSERTION FAILED: {name}\n  {detail}")
    print(f"  PASS  {name}" + (f"  [{detail}]" if detail else ""))


def rid_list(rids, limit=15):
    rids = sorted(int(r) for r in rids)
    return f"n={len(rids)} RIDs=[{', '.join(str(r) for r in rids[:limit])}" \
           f"{', ...' if len(rids) > limit else ''}]"


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def save_tsv(df, name):
    assert name not in PROTECTED, f"REFUSING to overwrite a protected frozen artifact: {name}"
    p = OUT_DIR / name
    df.to_csv(p, sep="\t", index=False)
    print(f"  saved -> {p.relative_to(PROJECT_ROOT)}   ({df.shape[0]} x {df.shape[1]})")
    return p


HASHES_BEFORE = {f: sha256(FREEZE_DIR / f) for f in PROTECTED}

actual = sha256(INPUT_PATH)
print(f"input : {INPUT_NAME}")
print(f"  expected sha256: {EXPECTED_SHA}")
print(f"  actual   sha256: {actual}")
if actual != EXPECTED_SHA:
    cands = sorted(FREEZE_DIR.glob("mci_survival_primary_cohort_*.tsv"))
    raise SystemExit(f"CHECKSUM MISMATCH. Do not continue silently. Candidates: "
                     f"{[c.name for c in cands]}. Investigate before rebuilding.")
check("input checksum matches the expected frozen-v1 primary cohort", actual == EXPECTED_SHA)
print(f"\npython {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")

## 1. Load the frozen primary cohort and verify its documented properties

In [ ]:
src = pd.read_csv(INPUT_PATH, sep="\t")
n, ncol = len(src), src.shape[1]
n_ev = int((src["event_indicator"] == 1).sum())
n_cn = int((src["event_indicator"] == 0).sum())

check("input has 401 unique participants", n == 401 and src["RID"].nunique() == 401,
      f"rows={n}, unique={src['RID'].nunique()}")
check("input has 85 events / 316 censored", (n_ev, n_cn) == (85, 316), f"{n_ev}/{n_cn}")
check("input has 78 columns", ncol == 78, f"cols={ncol}")
check("input duration strictly positive", bool((src["time_to_event_or_censor_days"] > 0).all()))
check("input event_indicator in {0,1}, no missing",
      set(src["event_indicator"].dropna().unique()) <= {0, 1} and src["event_indicator"].notna().all())

# ---- confirmed concept -> authoritative column map ----
SRC = dict(RID="RID", anchor_date="anchor_date",
           duration="time_to_event_or_censor_days", event="event_indicator",
           entry_age="entry_age", APOE4_COUNT="APOE4_COUNT", ptau217="ptau217",
           gfap="gfap", nfl="nfl", abeta42="abeta42", abeta40="abeta40",
           ratio="ratio_ab42_ab40",
           pre_anchor_dementia_n="pre_anchor_dementia_n",
           qc_dementia="qc_dementia_on_or_before_anchor")
for concept, col in SRC.items():
    check(f"source column '{col}' ({concept}) exists in the frozen cohort", col in src.columns)

# ---- the two prior-dementia QC fields must be UNAMBIGUOUS ----
dem_like = [c for c in src.columns if "dement" in c.lower() or "pre_anchor" in c.lower()
            or c.startswith("qc_")]
print("prior-dementia / QC candidate columns:", dem_like)
check("'pre_anchor_dementia_n' maps unambiguously (exact frozen column; count of pre-anchor dementia dx)",
      "pre_anchor_dementia_n" in src.columns
      and str(src["pre_anchor_dementia_n"].dtype).startswith("int"))
check("'qc_dementia_on_or_before_anchor' maps unambiguously (exact frozen boolean flag)",
      "qc_dementia_on_or_before_anchor" in src.columns)
print(f"  pre_anchor_dementia_n > 0 : {int((src['pre_anchor_dementia_n'] > 0).sum())} participants")
print(f"  qc_dementia flag == True  : {int(src['qc_dementia_on_or_before_anchor'].astype(bool).sum())} participants")
print(f"\nLoaded frozen primary cohort: {n} x {ncol}  ({n_ev} events, {n_cn} censored)")

## 2. Assemble the shared dataset — exactly 26 columns

One row per RID, **all 401 preserved**, sorted by RID.

### Derivations (match Person 2's published analysis)
```python
APOE4_CARRIER   = (APOE4_COUNT >= 1).astype(int)
log_ptau217     = np.log(ptau217)
log_gfap        = np.log(gfap)        # NA preserved where gfap missing
log_nfl         = np.log(nfl)         # NA preserved where nfl missing
ratio_ab42_ab40 = abeta42 / abeta40   # carried from the frozen column (identical by construction)
```

### Eligibility flags (per participant; complete-data membership for each analysis set)
`core := entry_age & APOE4_COUNT & ptau217` (complete for all 401). `amyloid := ratio_ab42_ab40`
present (equivalently both Aβ components valid). Each flag is `core AND the named markers present`,
coded `{0,1}`. They **materialize** each complete-case analysis set so downstream teams can filter
with one column and never re-derive availability.

In [ ]:
d = src.sort_values("RID").reset_index(drop=True)
out = pd.DataFrame()

# identification & provenance
out["RID"] = d["RID"]                                 # IDENTIFIER ONLY — never a predictor
out["anchor_date"] = d["anchor_date"]                 # provenance / harmonization only
# outcome (frozen; not recomputed)
out["time_to_event_or_censor_days"] = d["time_to_event_or_censor_days"]
out["event_indicator"] = d["event_indicator"].astype(int)
# core predictors
out["entry_age"] = d["entry_age"]
out["APOE4_COUNT"] = d["APOE4_COUNT"]
out["APOE4_CARRIER"] = (d["APOE4_COUNT"] >= 1).astype(int)
out["ptau217"] = d["ptau217"]
out["log_ptau217"] = np.log(d["ptau217"])
# blood biomarkers: raw + log interleaved
out["gfap"] = d["gfap"]
out["log_gfap"] = np.log(d["gfap"])
out["nfl"] = d["nfl"]
out["log_nfl"] = np.log(d["nfl"])
out["abeta42"] = d["abeta42"]
out["abeta40"] = d["abeta40"]
out["ratio_ab42_ab40"] = d["ratio_ab42_ab40"]         # == abeta42/abeta40 (asserted)
# prior-dementia QC fields (carried from frozen)
out["pre_anchor_dementia_n"] = d["pre_anchor_dementia_n"]
out["qc_dementia_on_or_before_anchor"] = d["qc_dementia_on_or_before_anchor"]

# ---- eligibility flags ----
core = d["entry_age"].notna() & d["APOE4_COUNT"].notna() & d["ptau217"].notna()
g, nf, am = d["gfap"].notna(), d["nfl"].notna(), d["ratio_ab42_ab40"].notna()
ELIG = {
    "eligible_core": core,
    "eligible_gfap": core & g,
    "eligible_nfl": core & nf,
    "eligible_amyloid": core & am,
    "eligible_gfap_nfl": core & g & nf,
    "eligible_gfap_amyloid": core & g & am,
    "eligible_nfl_amyloid": core & nf & am,
    "eligible_full_blood": core & g & nf & am,
}
for name, flag in ELIG.items():
    out[name] = flag.astype(int)

out = out[FINAL_COLUMNS]
print(f"Shared dataset: {out.shape[0]} x {out.shape[1]}")
for i, c in enumerate(out.columns, 1):
    print(f"  {i:2d}  {c:34s} {str(out[c].dtype):8s} missing={int(out[c].isna().sum())}")
out.head()

## 3. Validation

In [ ]:
# ---- exact column contract ----
check("output has EXACTLY the 26 specified columns, in the specified order",
      list(out.columns) == FINAL_COLUMNS,
      f"got {list(out.columns)}")

# ---- participant set preserved ----
check("401 rows, one per RID, no duplicates, sorted by RID",
      len(out) == 401 and out["RID"].is_unique and out["RID"].notna().all()
      and out["RID"].is_monotonic_increasing)
check("RID set == frozen primary cohort (no participant added or removed)",
      set(out["RID"]) == set(src["RID"]), rid_list(set(out["RID"]) ^ set(src["RID"])))

# ---- outcome carried faithfully ----
_m = out.merge(src[["RID", "time_to_event_or_censor_days", "event_indicator"]], on="RID",
               suffixes=("", "_s"))
check("duration + event carried byte-faithfully (not recomputed); durations strictly positive",
      bool((_m["time_to_event_or_censor_days"] == _m["time_to_event_or_censor_days_s"]).all())
      and bool((_m["event_indicator"] == _m["event_indicator_s"]).all())
      and bool((out["time_to_event_or_censor_days"] > 0).all()))
check("event counts preserved (85 / 316)",
      int((out["event_indicator"] == 1).sum()) == 85 and int((out["event_indicator"] == 0).sum()) == 316)

# ---- core predictors complete ----
for c in ["entry_age", "APOE4_COUNT", "APOE4_CARRIER", "ptau217", "log_ptau217"]:
    check(f"core predictor '{c}' complete for all 401", out[c].notna().all())

# ---- derivations faithful ----
check("APOE4_CARRIER == (APOE4_COUNT>=1), coded {0,1}",
      bool((out["APOE4_CARRIER"] == (d["APOE4_COUNT"] >= 1).astype(int)).all())
      and set(out["APOE4_CARRIER"].unique()) <= {0, 1})
# strict tolerances throughout: np.allclose defaults (rtol=1e-5) are far too loose to detect a
# genuine derivation discrepancy. Full diagnostics for the amyloid ratio are in section 3b.
for lc, rc in [("log_ptau217", "ptau217"), ("log_gfap", "gfap"), ("log_nfl", "nfl")]:
    check(f"{lc} == np.log({rc}) at rtol=1e-10, atol=1e-12; NA exactly where {rc} is NA; no +/-inf",
          bool(np.allclose(out[lc], np.log(d[rc]), rtol=1e-10, atol=1e-12, equal_nan=True))
          and bool((out[lc].isna() == d[rc].isna()).all())
          and not bool(np.isinf(out[lc].fillna(0)).any()))
check("ratio_ab42_ab40 == abeta42/abeta40 at rtol=1e-10, atol=1e-12 (carried from frozen)",
      bool(np.allclose(out["ratio_ab42_ab40"], d["abeta42"] / d["abeta40"],
                       rtol=1e-10, atol=1e-12, equal_nan=True)))

# ---- QC fields carried faithfully ----
check("pre_anchor_dementia_n carried byte-faithfully from frozen",
      bool((out["pre_anchor_dementia_n"].values == d["pre_anchor_dementia_n"].values).all()))
check("qc_dementia_on_or_before_anchor carried byte-faithfully from frozen",
      bool((out["qc_dementia_on_or_before_anchor"].astype(bool).values
            == d["qc_dementia_on_or_before_anchor"].astype(bool).values).all()))
check("prior-dementia flag agrees with the count field (flag <=> n>0)",
      bool((out["qc_dementia_on_or_before_anchor"].astype(bool)
            == (out["pre_anchor_dementia_n"] > 0)).all()),
      f"flagged={int(out['qc_dementia_on_or_before_anchor'].astype(bool).sum())}")

# ---- eligibility flags correct ----
# recomputed from the SPEC definitions (log-scale predictors), then compared to the written flags
_core = out["entry_age"].notna() & out["APOE4_COUNT"].notna() & out["log_ptau217"].notna()
SPEC_ELIG = {
    "eligible_core": _core,
    "eligible_gfap": _core & out["log_gfap"].notna(),
    "eligible_nfl": _core & out["log_nfl"].notna(),
    "eligible_amyloid": _core & out["ratio_ab42_ab40"].notna(),
    "eligible_gfap_nfl": _core & out["log_gfap"].notna() & out["log_nfl"].notna(),
    "eligible_gfap_amyloid": _core & out["log_gfap"].notna() & out["ratio_ab42_ab40"].notna(),
    "eligible_nfl_amyloid": _core & out["log_nfl"].notna() & out["ratio_ab42_ab40"].notna(),
    "eligible_full_blood": (_core & out["log_gfap"].notna() & out["log_nfl"].notna()
                            & out["ratio_ab42_ab40"].notna()),
}
# (participants, events, censored) expected for every flag
EXP_ELIG = {"eligible_core": (401, 85, 316), "eligible_gfap": (361, 85, 276),
            "eligible_nfl": (361, 85, 276), "eligible_amyloid": (399, 83, 316),
            "eligible_gfap_nfl": (361, 85, 276), "eligible_gfap_amyloid": (359, 83, 276),
            "eligible_nfl_amyloid": (359, 83, 276), "eligible_full_blood": (359, 83, 276)}
for name, (exp_n, exp_ev, exp_cn) in EXP_ELIG.items():
    got_n = int(out[name].sum())
    got_ev = int(out.loc[out[name] == 1, "event_indicator"].sum())
    check(f"{name}: {exp_n} participants / {exp_ev} events / {exp_cn} censored "
          f"(recomputed from nonmissingness, never assigned)",
          (got_n, got_ev, got_n - got_ev) == (exp_n, exp_ev, exp_cn),
          f"got {got_n}/{got_ev}/{got_n - got_ev}")
    check(f"{name} matches the spec definition exactly and is coded strictly {{0,1}}",
          bool((out[name] == SPEC_ELIG[name].astype(int)).all())
          and set(out[name].unique()) <= {0, 1}
          and str(out[name].dtype).startswith("int"))
check("eligibility flags are properly nested (full_blood subset-of gfap_nfl subset-of gfap/nfl subset-of core)",
      bool((out["eligible_full_blood"] <= out["eligible_gfap_nfl"]).all())
      and bool((out["eligible_gfap_nfl"] <= out["eligible_gfap"]).all())
      and bool((out["eligible_gfap"] <= out["eligible_core"]).all()))
check("eligible_core == 1 for all 401 (core predictors complete by construction)",
      bool((out["eligible_core"] == 1).all()))

# ---- NO imputation ----
EXPECT_MISS = {"gfap": 40, "log_gfap": 40, "nfl": 40, "log_nfl": 40,
               "abeta42": 2, "abeta40": 2, "ratio_ab42_ab40": 2}
for c, exp in EXPECT_MISS.items():
    check(f"no imputation: '{c}' has {exp} NA, matching frozen", int(out[c].isna().sum()) == exp)
check("dataset NOT reduced to any complete-case subset (all 401 retained)", len(out) == 401)

# ---- NO scaling: raw carried columns byte-identical to frozen ----
for c in ["entry_age", "APOE4_COUNT", "ptau217", "gfap", "nfl", "abeta42", "abeta40",
          "ratio_ab42_ab40", "pre_anchor_dementia_n"]:
    a = out.set_index("RID")[c].sort_index()
    b = src.set_index("RID")[c].sort_index()
    check(f"no scaling: '{c}' byte-identical to frozen", a.equals(b))

# ---- prohibited content absent ----
check("no cognitive score present",
      not any(any(k in col for k in ["CDRSB", "MMSCORE", "MOCA", "FAQTOTAL", "cdrsb", "mmse",
                                     "moca", "faq"]) for col in out.columns))
check("no CV-fold column present",
      not any("fold" in c.lower() or c.lower().startswith("cv_") for c in out.columns))

### 3b. Missingness-structure and prior-dementia assertions

Section 3 covers the column contract, the derivations, and the NA *counts*. This section asserts the
**structure** of the missingness and the prior-dementia QC facts explicitly, rather than leaving
them to prose.

**Amyloid-ratio agreement is checked at `rtol=1e-10, atol=1e-12`** — not at `np.allclose`'s default
`rtol=1e-5`, which is several orders of magnitude too loose to detect a real ratio-recomputation
discrepancy.

#### Prior dementia: participants vs. diagnosis records

These two levels are routinely conflated, so state them exactly:

| Level | Count | Meaning |
|---|---:|---|
| Anchor cohort **A** | **8** | eight **participants** with ≥1 dementia diagnosis strictly *before* their MCI anchor (Build 1 §6.1) |
| Frozen primary cohort **C** | **4** | four of those eight survive the cohort-C filters: RIDs **467, 4115, 4506, 4706** |
| Diagnosis **records** | **12** | those four carry 12 pre-anchor dementia diagnosis records in total (5 + 4 + 2 + 1) |
| Events among the four | **1** | RID 467 only |

The previously circulated "8" is a count of **participants in cohort A**. It is *not* a record count,
and *not* a count within cohort C. There is no reading under which it is "8 records across 4
participants" — that framing is wrong at both levels and is not preserved anywhere in this build.

- `pre_anchor_dementia_n` — a **participant-level** field whose **value** is a diagnosis-**record** count.
- `qc_dementia_on_or_before_anchor` — a **participant-level binary flag** (`True ⇔ pre_anchor_dementia_n > 0`).

Neither field may be used as a model predictor, and neither may be used to filter this shared
dataset. They exist for QC traceability and for an explicitly-labelled sensitivity analysis only.
All four participants are **retained**; adjudication remains pending.

In [ ]:
# =====================================================================================
# 3b. Explicit missingness-structure + prior-dementia assertions
# =====================================================================================

# ---- 1. GFAP and NfL availability masks are IDENTICAL (same Quanterix draw) ----
gfap_missing = out["gfap"].isna()
nfl_missing = out["nfl"].isna()
check("GFAP and NfL availability masks are IDENTICAL (same 40 participants missing both)",
      bool((gfap_missing == nfl_missing).all()) and int(gfap_missing.sum()) == 40,
      f"gfap_missing={int(gfap_missing.sum())}, nfl_missing={int(nfl_missing.sum())}, "
      f"symmetric_difference={int((gfap_missing ^ nfl_missing).sum())}")
check("log_gfap / log_nfl inherit exactly the raw availability masks",
      bool((out["log_gfap"].isna() == gfap_missing).all())
      and bool((out["log_nfl"].isna() == nfl_missing).all()))

# ---- 2. EXACTLY two amyloid-missing participants ----
amyloid_missing = out["ratio_ab42_ab40"].isna()
amyloid_missing_rids = sorted(int(r) for r in out.loc[amyloid_missing, "RID"])
check("EXACTLY 2 participants are missing the amyloid ratio",
      int(amyloid_missing.sum()) == 2, f"RIDs={amyloid_missing_rids}")
check("abeta42 / abeta40 / ratio share one identical missingness mask",
      bool((out["abeta42"].isna() == amyloid_missing).all())
      and bool((out["abeta40"].isna() == amyloid_missing).all()))

# ---- 3. BOTH amyloid-missing participants are EVENTS ----
check("BOTH amyloid-missing participants are events (this is why 85 -> 83)",
      bool((out.loc[amyloid_missing, "event_indicator"] == 1).all())
      and int(out.loc[amyloid_missing, "event_indicator"].sum()) == 2,
      f"RIDs={amyloid_missing_rids}, "
      f"events={out.loc[amyloid_missing, 'event_indicator'].tolist()}")

# ---- 4. NO OVERLAP between the amyloid-missing and GFAP/NfL-missing groups ----
overlap = amyloid_missing & gfap_missing
check("NO OVERLAP between amyloid-missing and GFAP/NfL-missing participants",
      int(overlap.sum()) == 0,
      f"overlap_n={int(overlap.sum())}; this is why full_blood N = 401 - 40 - 2 = 359")
check("full_blood N is exactly explained by two disjoint missingness groups (401-40-2=359)",
      int(out["eligible_full_blood"].sum()) == 401 - int(gfap_missing.sum()) - int(amyloid_missing.sum()))

# ---- 5. Prior-dementia flag sums to EXACTLY 4 participants ----
qc_flag = out["qc_dementia_on_or_before_anchor"].astype(bool)
qc_rids = sorted(int(r) for r in out.loc[qc_flag, "RID"])
check("qc_dementia_on_or_before_anchor sums to EXACTLY 4 (participants, not records)",
      int(qc_flag.sum()) == 4, f"RIDs={qc_rids}")

# ---- 6. EXACTLY 1 event among those four participants ----
qc_events = int(out.loc[qc_flag, "event_indicator"].sum())
check("EXACTLY 1 event among the 4 prior-dementia participants",
      qc_events == 1,
      f"event RIDs={sorted(int(r) for r in out.loc[qc_flag & (out['event_indicator'] == 1), 'RID'])}")

# ---- 6b. record count vs participant count, asserted (not merely asserted in prose) ----
qc_records = int(out.loc[qc_flag, "pre_anchor_dementia_n"].sum())
check("the 4 prior-dementia PARTICIPANTS carry 12 diagnosis RECORDS in total (record != participant)",
      qc_records == 12 and int(qc_flag.sum()) == 4,
      f"participants={int(qc_flag.sum())} (RIDs={qc_rids}), records={qc_records} "
      f"(per-participant={out.loc[qc_flag, 'pre_anchor_dementia_n'].tolist()})")
check("every unflagged participant has pre_anchor_dementia_n == 0",
      bool((out.loc[~qc_flag, "pre_anchor_dementia_n"] == 0).all()))

# ---- 7. Amyloid-ratio agreement at STRICT tolerance (NOT np.allclose defaults) ----
RTOL, ATOL = 1e-10, 1e-12
cmp_mask = out["abeta42"].notna() & out["abeta40"].notna() & out["ratio_ab42_ab40"].notna()
recomputed_ratio = out.loc[cmp_mask, "abeta42"] / out.loc[cmp_mask, "abeta40"]
frozen_ratio = out.loc[cmp_mask, "ratio_ab42_ab40"]
abs_diff = (recomputed_ratio - frozen_ratio).abs()
rel_diff = abs_diff / frozen_ratio.abs()
discrepant = abs_diff > (ATOL + RTOL * frozen_ratio.abs())

if bool(discrepant.any()):
    # Do NOT silently pick a version — surface everything the spec asks for and stop.
    bad = out.loc[cmp_mask].loc[discrepant, "RID"]
    raise AssertionError(
        f"AMYLOID RATIO DISCREPANCY — recomputed != frozen.\n"
        f"  n_discrepancies : {int(discrepant.sum())}\n"
        f"  max abs diff    : {abs_diff.max():.6e}\n"
        f"  max rel diff    : {rel_diff.max():.6e}\n"
        f"  example RIDs    : {sorted(int(r) for r in bad)[:15]}\n"
        f"  Neither version may be chosen silently. Investigate the source mapping.")

check(f"recomputed amyloid ratio == frozen at rtol={RTOL:g}, atol={ATOL:g} (NOT allclose defaults)",
      bool(np.allclose(recomputed_ratio, frozen_ratio, rtol=RTOL, atol=ATOL)),
      f"n_compared={int(cmp_mask.sum())}, n_discrepancies=0, "
      f"max_abs_diff={abs_diff.max():.3e}, max_rel_diff={rel_diff.max():.3e}")

# ---- 8. Every observed log/ratio input is finite and strictly positive ----
for col in ["ptau217", "gfap", "nfl", "abeta42", "abeta40"]:
    obs = out.loc[out[col].notna(), col]
    check(f"every observed '{col}' is finite and strictly positive (safe for log/ratio)",
          bool(np.isfinite(obs).all()) and bool((obs > 0).all()),
          f"n={len(obs)}, min={obs.min():.6g}")

print(f"\n  amyloid-missing RIDs         : {amyloid_missing_rids}  (both events)")
print(f"  GFAP/NfL-missing participants: {int(gfap_missing.sum())}  (identical masks)")
print(f"  overlap of the two groups    : {int(overlap.sum())}  ->  full_blood = 401-40-2 = "
      f"{int(out['eligible_full_blood'].sum())}")
print(f"  prior-dementia participants  : {int(qc_flag.sum())} (RIDs {qc_rids}) "
      f"carrying {qc_records} diagnosis records; {qc_events} event")

## 4. Analysis-set counts

One row per `eligible_*` flag, computed straight from the flags — the shared file is never subset
here. This documents the **sample cost** of each feature combination.

Fixed schema: `analysis_set`, `eligibility_flag`, `required_features`, `n_participants`,
`n_events`, `n_censored`, `participant_fraction`, `event_fraction`, `notes`.

```python
participant_fraction = n_participants / 401
event_fraction       = n_events / 85
n_censored           = n_participants - n_events
```

Fractions are stored as **full-precision numerics** (e.g. `0.9002493765586035`), never rounded and
never formatted as percentage strings.

In [ ]:
# Schema is fixed by spec: analysis_set, eligibility_flag, required_features, n_participants,
# n_events, n_censored, participant_fraction, event_fraction, notes.
# Fractions are stored as full-precision numerics — NOT rounded, NOT formatted percentage strings.
COHORT_N, COHORT_EVENTS = 401, 85

SET_META = {
    "eligible_core": ("entry_age + APOE4_COUNT + log_ptau217",
                      "Core primary model (M1). Equals the frozen primary cohort exactly."),
    "eligible_gfap": ("entry_age + APOE4_COUNT + log_ptau217 + log_gfap",
                      "Incremental GFAP comparison; refit M1 on this same 361-participant set."),
    "eligible_nfl": ("entry_age + APOE4_COUNT + log_ptau217 + log_nfl",
                     "Incremental NfL comparison; refit M1 on this same 361-participant set."),
    "eligible_amyloid": ("entry_age + APOE4_COUNT + log_ptau217 + ratio_ab42_ab40",
                         "Incremental amyloid comparison; refit M1 on this same 399-participant set."),
    "eligible_gfap_nfl": ("entry_age + APOE4_COUNT + log_ptau217 + log_gfap + log_nfl",
                          "GFAP and NfL share one availability mask, so this equals eligible_gfap "
                          "and eligible_nfl."),
    "eligible_gfap_amyloid": ("entry_age + APOE4_COUNT + log_ptau217 + log_gfap + ratio_ab42_ab40",
                              "Two-marker extension."),
    "eligible_nfl_amyloid": ("entry_age + APOE4_COUNT + log_ptau217 + log_nfl + ratio_ab42_ab40",
                             "Two-marker extension."),
    "eligible_full_blood": ("entry_age + APOE4_COUNT + log_ptau217 + log_gfap + log_nfl "
                            "+ ratio_ab42_ab40",
                            "Common subset for head-to-head ranking of ALL biomarker combinations."),
}

rows = []
for flag, (required_features, note) in SET_META.items():
    m = out[flag] == 1
    n_participants = int(m.sum())
    n_events = int((out.loc[m, "event_indicator"] == 1).sum())
    n_censored = n_participants - n_events           # spec: n_censored = n_participants - n_events
    rows.append({
        "analysis_set": flag,
        "eligibility_flag": flag,
        "required_features": required_features,
        "n_participants": n_participants,
        "n_events": n_events,
        "n_censored": n_censored,
        "participant_fraction": n_participants / COHORT_N,    # full precision, not a percentage
        "event_fraction": n_events / COHORT_EVENTS,           # full precision, not a percentage
        "notes": note,
    })
counts = pd.DataFrame(rows)

COUNTS_SCHEMA = ["analysis_set", "eligibility_flag", "required_features", "n_participants",
                 "n_events", "n_censored", "participant_fraction", "event_fraction", "notes"]
counts = counts[COUNTS_SCHEMA]

# ---- schema + arithmetic assertions ----
check("counts table uses EXACTLY the specified 9-column schema, in order",
      list(counts.columns) == COUNTS_SCHEMA, f"got {list(counts.columns)}")
check("counts table has one row for every eligibility flag (8)",
      len(counts) == 8 and set(counts["analysis_set"]) == set(ELIG),
      f"rows={len(counts)}")
for _, r in counts.iterrows():
    flag = r["eligibility_flag"]
    check(f"counts row '{flag}': N == flag sum, censored == N - events, fractions exact",
          int(r["n_participants"]) == int(out[flag].sum())
          and int(r["n_censored"]) == int(r["n_participants"]) - int(r["n_events"])
          and r["participant_fraction"] == int(r["n_participants"]) / COHORT_N
          and r["event_fraction"] == int(r["n_events"]) / COHORT_EVENTS)
check("fractions stored as full-precision floats (not rounded, not percentage strings)",
      counts["participant_fraction"].map(lambda v: isinstance(v, float)).all()
      and counts["event_fraction"].map(lambda v: isinstance(v, float)).all()
      and abs(float(counts.loc[counts["analysis_set"] == "eligible_gfap",
                               "participant_fraction"].iloc[0]) - 361 / 401) == 0.0,
      f"eligible_gfap participant_fraction = "
      f"{counts.loc[counts['analysis_set'] == 'eligible_gfap', 'participant_fraction'].iloc[0]!r}")
check("core analysis set reproduces the frozen primary cohort (401 / 85 / 316)",
      tuple(counts.loc[counts["analysis_set"] == "eligible_core",
                       ["n_participants", "n_events", "n_censored"]].iloc[0]) == (401, 85, 316))
check("full_blood analysis set = 359 / 83 / 276",
      tuple(counts.loc[counts["analysis_set"] == "eligible_full_blood",
                       ["n_participants", "n_events", "n_censored"]].iloc[0]) == (359, 83, 276))

print(counts[["analysis_set", "n_participants", "n_events", "n_censored",
              "participant_fraction", "event_fraction"]].to_string(index=False))

## 5. Data dictionary — one row per shared-dataset column (26)

Fixed schema: `output_column`, `source_column`, `role`, `description`, `dtype`, `units`,
`derivation`, `allowed_missing`, `expected_missing_n`, `modeling_rule`, `provenance_notes`.

Roles distinguish identifiers, provenance fields, the survival duration, the survival event, core
predictors, optional predictors, **raw audit variables**, QC-only fields, and analysis-set flags.

Three prohibitions are stated explicitly on the rows they apply to:

- **Raw biomarker fields are retained for auditability**, not for modeling — never put the raw and
  the log form of the same biomarker in one model.
- **QC fields are never predictors** and never a filter for the shared primary dataset.
- **Analysis-set flags are never predictors** — they select rows, nothing more.

`APOE4_COUNT` and `APOE4_CARRIER` also carry a mutual same-model prohibition.

In [ ]:
# Fixed dictionary schema (spec): output_column, source_column, role, description, dtype, units,
# derivation, allowed_missing, expected_missing_n, modeling_rule, provenance_notes.
DICT_SCHEMA = ["output_column", "source_column", "role", "description", "dtype", "units",
               "derivation", "allowed_missing", "expected_missing_n", "modeling_rule",
               "provenance_notes"]

FZ = "mci_survival_primary_cohort_v1.tsv (frozen v1)"
CARRIED = f"carried byte-for-byte from {FZ}"

# Reusable modeling rules -------------------------------------------------------------------
RULE_NEVER = "NEVER a predictor."
RULE_QC = ("QC / sensitivity only. NEVER a predictor, and NEVER a filter for the shared primary "
           "dataset. Use only in an explicitly-labelled sensitivity analysis.")
RULE_FLAG = ("Analysis-set selector. NEVER a predictor — use it only to subset rows to the "
             "participants who have the required features.")
RULE_RAW_AUDIT = ("Retained for AUDITABILITY / traceability. Prefer the log form for modeling; "
                  "NEVER put the raw and log form of the same biomarker in one model.")


def R(col, source_column, role, description, units, derivation, allowed_missing,
      modeling_rule, provenance_notes):
    return dict(output_column=col, source_column=source_column, role=role, description=description,
                units=units, derivation=derivation, allowed_missing=allowed_missing,
                modeling_rule=modeling_rule, provenance_notes=provenance_notes)


DD = [
 # --- identifier & provenance ---
 R("RID", "RID", "IDENTIFIER", "ADNI participant identifier; primary key of this dataset.",
   "integer id", CARRIED, "no",
   RULE_NEVER + " It carries site/enrolment-order information and would leak.",
   "One row per RID; 401 unique, sorted ascending."),
 R("anchor_date", "anchor_date", "PROVENANCE",
   "Date of the plasma draw that anchors follow-up for this participant.", "date (YYYY-MM-DD)",
   CARRIED, "no",
   RULE_NEVER + " Provenance and NACC harmonization only; calendar time is not a study variable.",
   "Defines t=0; not recomputed here."),

 # --- survival outcome ---
 R("time_to_event_or_censor_days", "time_to_event_or_censor_days", "SURVIVAL DURATION",
   "Days from anchor_date to incident dementia, or to last dementia-free follow-up if censored.",
   "days", CARRIED + " (not recomputed)", "no",
   "Duration term of the survival model. Always pair with event_indicator.",
   "Strictly positive for all 401; frozen by Build 1."),
 R("event_indicator", "event_indicator", "SURVIVAL EVENT",
   "1 = incident all-cause dementia during follow-up; 0 = censored dementia-free.", "0/1",
   CARRIED + " (not recomputed)", "no",
   "Event term of the survival model. 0 means CENSORED, not 'stable' — never treat it as a "
   "binary progression label.",
   "85 events / 316 censored; frozen by Build 1."),

 # --- core predictors ---
 R("entry_age", "entry_age", "CORE PREDICTOR", "Age at study entry.", "years", CARRIED, "no",
   "Core predictor in M0 and M1.",
   "UNDATED upstream: age at ENTRY, not age at the anchor draw. Documented limitation."),
 R("APOE4_COUNT", "APOE4_COUNT", "CORE PREDICTOR", "Number of APOE e4 alleles.", "alleles (0/1/2)",
   CARRIED, "no",
   "Core predictor in M0 and M1. NEVER include together with APOE4_CARRIER — they encode the "
   "same variable and are collinear by construction.",
   "Complete in the primary cohort by cohort-C construction."),
 R("APOE4_CARRIER", "APOE4_COUNT", "CORE PREDICTOR (derived)",
   "Binary APOE e4 carrier status.", "0/1", "(APOE4_COUNT >= 1).astype(int)", "no",
   "Alternative coding of APOE4_COUNT for sensitivity analysis. NEVER include together with "
   "APOE4_COUNT in the same model.",
   "Derived in this build; deterministic function of APOE4_COUNT."),
 R("ptau217", "ptau217", "RAW AUDIT VARIABLE", "Plasma phosphorylated tau 217, raw scale.",
   "pg/mL", CARRIED, "no",
   RULE_RAW_AUDIT + " log_ptau217 is the modeling form used in M1.",
   "Fujirebio platform. Strictly positive for all 401 (asserted)."),
 R("log_ptau217", "ptau217", "CORE PREDICTOR (derived)",
   "Natural log of plasma p-tau217.", "log(pg/mL)", "np.log(ptau217)  [natural log]", "no",
   "The p-tau217 term of M1. NEVER include together with raw ptau217.",
   "numpy.log; every input finite and strictly positive (asserted)."),

 # --- optional blood biomarkers ---
 R("gfap", "gfap", "RAW AUDIT VARIABLE", "Plasma glial fibrillary acidic protein, raw scale.",
   "pg/mL", CARRIED, "yes — optional biomarker",
   RULE_RAW_AUDIT + " log_gfap is the modeling form.",
   "Quanterix panel. Missing for exactly the same 40 participants as nfl."),
 R("log_gfap", "gfap", "OPTIONAL PREDICTOR (derived)", "Natural log of plasma GFAP.",
   "log(pg/mL)", "np.log(gfap)  [natural log; NA preserved where gfap is NA]",
   "yes — optional biomarker",
   "Candidate extension to M1. NEVER include together with raw gfap.",
   "numpy.log; NA mask identical to gfap (asserted)."),
 R("nfl", "nfl", "RAW AUDIT VARIABLE", "Plasma neurofilament light chain, raw scale.", "pg/mL",
   CARRIED, "yes — optional biomarker",
   RULE_RAW_AUDIT + " log_nfl is the modeling form.",
   "Quanterix panel. Missing for exactly the same 40 participants as gfap."),
 R("log_nfl", "nfl", "OPTIONAL PREDICTOR (derived)", "Natural log of plasma NfL.", "log(pg/mL)",
   "np.log(nfl)  [natural log; NA preserved where nfl is NA]", "yes — optional biomarker",
   "Candidate extension to M1. NEVER include together with raw nfl.",
   "numpy.log; NA mask identical to nfl (asserted)."),
 R("abeta42", "abeta42", "RAW AUDIT VARIABLE", "Plasma amyloid-beta 42, raw scale.", "pg/mL",
   CARRIED, "yes — optional biomarker",
   "Retained for AUDITABILITY as the ratio numerator. Model the ratio, not the component; "
   "NEVER include a component alongside ratio_ab42_ab40.",
   "Fujirebio platform; same assay row as abeta40."),
 R("abeta40", "abeta40", "RAW AUDIT VARIABLE", "Plasma amyloid-beta 40, raw scale.", "pg/mL",
   CARRIED, "yes — optional biomarker",
   "Retained for AUDITABILITY as the ratio denominator. Model the ratio, not the component; "
   "NEVER include a component alongside ratio_ab42_ab40.",
   "Fujirebio platform; strictly positive wherever observed (asserted)."),
 R("ratio_ab42_ab40", "ratio_ab42_ab40", "OPTIONAL PREDICTOR",
   "Plasma amyloid-beta 42 to 40 ratio.", "ratio (unitless)",
   "abeta42 / abeta40  (carried from the frozen column; recomputation verified to agree at "
   "rtol=1e-10, atol=1e-12)", "yes — optional biomarker",
   "Candidate extension to M1. NEVER include together with abeta42 or abeta40.",
   "Both components come from the same assay row; NA where either component is NA."),

 # --- prior-dementia QC (NOT predictors) ---
 R("pre_anchor_dementia_n", "pre_anchor_dementia_n", "QC ONLY",
   "PARTICIPANT-level field whose VALUE is a count of dementia diagnosis RECORDS dated strictly "
   "before this participant's anchor. A record count, not a participant count.",
   "count of diagnosis records", CARRIED, "no",
   RULE_QC,
   "Build 1 §6.1 identified 8 PARTICIPANTS in anchor cohort A with pre-anchor dementia; 4 of "
   "them remain in this frozen primary cohort (RIDs 467, 4115, 4506, 4706), carrying 12 "
   "diagnosis RECORDS in total (5+4+2+1). The circulated '8' is a participant count in cohort "
   "A, never a record count. All 4 are RETAINED; adjudication pending."),
 R("qc_dementia_on_or_before_anchor", "qc_dementia_on_or_before_anchor", "QC ONLY",
   "PARTICIPANT-level binary flag: this participant has >= 1 dementia diagnosis on or before "
   "the anchor. True for exactly 4 participants.", "boolean", CARRIED, "no",
   RULE_QC,
   "Equivalent to pre_anchor_dementia_n > 0 (asserted). Exactly 1 of the 4 flagged "
   "participants is an event (RID 467). Retained, not excluded."),
]

# --- analysis-set flags (generated; identical structure) ---
FLAG_META = {
    "eligible_core": ("core predictors (entry_age, APOE4_COUNT, log_ptau217)", 401),
    "eligible_gfap": ("core + log_gfap", 361),
    "eligible_nfl": ("core + log_nfl", 361),
    "eligible_amyloid": ("core + ratio_ab42_ab40", 399),
    "eligible_gfap_nfl": ("core + log_gfap + log_nfl", 361),
    "eligible_gfap_amyloid": ("core + log_gfap + ratio_ab42_ab40", 359),
    "eligible_nfl_amyloid": ("core + log_nfl + ratio_ab42_ab40", 359),
    "eligible_full_blood": ("core + log_gfap + log_nfl + ratio_ab42_ab40", 359),
}
for flag, (features, n_elig) in FLAG_META.items():
    DD.append(R(
        flag, "(derived from feature availability)", "ANALYSIS-SET FLAG",
        f"1 if the participant has complete data for {features}; else 0. {n_elig} eligible.",
        "0/1",
        f"({features.replace('core', 'entry_age & APOE4_COUNT & log_ptau217')}) all notna(), "
        f"cast to int",
        "no",
        RULE_FLAG,
        "Derived from observed nonmissingness only — never manually assigned. Counts reconciled "
        "against mci_survival_analysis_set_counts_v1.tsv."))

dd = pd.DataFrame(DD)
dd["dtype"] = dd["output_column"].map(lambda c: str(out[c].dtype))
dd["expected_missing_n"] = dd["output_column"].map(lambda c: int(out[c].isna().sum()))
dd = dd[DICT_SCHEMA]

# ---- dictionary assertions ----
check("data dictionary uses EXACTLY the specified 11-column schema, in order",
      list(dd.columns) == DICT_SCHEMA, f"got {list(dd.columns)}")
check("data dictionary covers EVERY output column exactly once, in the exact output order",
      list(dd["output_column"]) == list(out.columns) == FINAL_COLUMNS)
check("dictionary expected_missing_n matches the dataset exactly",
      bool((dd.set_index("output_column")["expected_missing_n"].reindex(out.columns).values
            == out.isna().sum().values).all()))
check("allowed_missing == 'no' exactly where the column is complete in all 401",
      bool((((dd["allowed_missing"] == "no").values)
            == (dd["expected_missing_n"] == 0).values).all()))
check("every source_column is a real frozen column or explicitly marked derived",
      bool(dd["source_column"].map(
          lambda s: s in src.columns or s.startswith("(derived")).all()),
      f"unmapped={sorted(set(dd['source_column']) - set(src.columns) - {'(derived from feature availability)'})}")
check("every row states a modeling_rule and a description (no blanks)",
      bool((dd["modeling_rule"].str.len() > 0).all()) and bool((dd["description"].str.len() > 0).all()))
check("QC fields and analysis-set flags are documented as NEVER predictors",
      bool(dd.loc[dd["role"] == "QC ONLY", "modeling_rule"].str.contains("NEVER a predictor").all())
      and bool(dd.loc[dd["role"] == "ANALYSIS-SET FLAG",
                      "modeling_rule"].str.contains("NEVER a predictor").all()))
check("raw biomarker fields are documented as retained for auditability, "
      "with the raw-vs-log same-model prohibition",
      bool(dd.loc[dd["role"] == "RAW AUDIT VARIABLE", "modeling_rule"]
           .str.contains("AUDITABILITY").all())
      and bool(dd.loc[dd["output_column"].isin(["ptau217", "gfap", "nfl"]), "modeling_rule"]
               .str.contains("NEVER put the raw and log form").all()))
check("APOE4_COUNT / APOE4_CARRIER same-model prohibition documented on both rows",
      bool(dd.loc[dd["output_column"].isin(["APOE4_COUNT", "APOE4_CARRIER"]), "modeling_rule"]
           .str.contains("NEVER include together with").all()))
check("prior-dementia rows distinguish RECORD count from PARTICIPANT count",
      bool(dd.loc[dd["output_column"] == "pre_anchor_dementia_n", "description"]
           .str.contains("record").all())
      and bool(dd.loc[dd["output_column"] == "pre_anchor_dementia_n", "provenance_notes"]
               .str.contains("12").all()))
check("all 7 distinct roles present (identifier, provenance, duration, event, core, optional, "
      "raw audit, QC, analysis-set flag)",
      set(dd["role"]) == {"IDENTIFIER", "PROVENANCE", "SURVIVAL DURATION", "SURVIVAL EVENT",
                          "CORE PREDICTOR", "CORE PREDICTOR (derived)", "OPTIONAL PREDICTOR",
                          "OPTIONAL PREDICTOR (derived)", "RAW AUDIT VARIABLE", "QC ONLY",
                          "ANALYSIS-SET FLAG"},
      f"got {sorted(set(dd['role']))}")

print(f"Data dictionary: {len(dd)} rows x {dd.shape[1]} columns")
print(dd[["output_column", "source_column", "role", "expected_missing_n"]].to_string(index=False))

## 6. Write outputs + manifest; verify frozen v1 artifacts byte-identical

The manifest is **deterministic**: it contains no build-time clock reading and no volatile
git dirty-state field, so it hashes identically across clean-kernel executions. It records the
HEAD commit from `git rev-parse HEAD`, the verified input path and checksum, the complete
source-to-output column mapping, the exact derived-column formulas, `numpy.log` as the natural-log
implementation, the Aβ42/Aβ40 ratio definition, counts for every eligibility flag, the reserved
downstream CV seed `20260722`, the full no-imputation / no-scaling / no-cognition / no-model-fitting
/ no-CV policy block, and the software versions.

It hashes **every finalized non-manifest artifact** — the dataset, the data dictionary, the counts
table, **and the build report**. A manifest cannot contain its own final SHA-256, so that one hash
is verified externally with `shasum -a 256`.

In [ ]:
def git(*a):
    try:
        return subprocess.run(["git", *a], cwd=PROJECT_ROOT, capture_output=True, text=True,
                              check=True).stdout.strip()
    except Exception:
        return None


REPORT_REL = "reports/mci_survival_build4_shared_modeling_dataset_v1.md"
REPORT_PATH = PROJECT_ROOT / REPORT_REL

ds_path = save_tsv(out, f"mci_survival_shared_modeling_dataset_{VERSION}.tsv")
dd_path = save_tsv(dd, f"mci_survival_shared_modeling_data_dictionary_{VERSION}.tsv")
ct_path = save_tsv(counts, f"mci_survival_analysis_set_counts_{VERSION}.tsv")

# Every finalized NON-manifest artifact is hashed here, including the build report.
FINALIZED = [
    (f"outputs/03a_mci_survival_shared_modeling/{ds_path.name}", ds_path, out.shape),
    (f"outputs/03a_mci_survival_shared_modeling/{dd_path.name}", dd_path, dd.shape),
    (f"outputs/03a_mci_survival_shared_modeling/{ct_path.name}", ct_path, counts.shape),
    (REPORT_REL, REPORT_PATH, None),
]
check("the build report exists and can be hashed into the manifest", REPORT_PATH.is_file(),
      REPORT_REL)

# ---- complete source-to-output column mapping (26 outputs) ----
SOURCE_TO_OUTPUT = {r["output_column"]: r["source_column"] for r in DD}

# ---- exact derived-column formulas ----
DERIVED_FORMULAS = {
    "APOE4_CARRIER": "(APOE4_COUNT >= 1).astype(int)",
    "log_ptau217": "np.log(ptau217)",
    "log_gfap": "np.log(gfap)",
    "log_nfl": "np.log(nfl)",
    "ratio_ab42_ab40": "abeta42 / abeta40",
    "eligible_core": "(entry_age.notna() & APOE4_COUNT.notna() & log_ptau217.notna()).astype(int)",
    "eligible_gfap": "(eligible_core & log_gfap.notna()).astype(int)",
    "eligible_nfl": "(eligible_core & log_nfl.notna()).astype(int)",
    "eligible_amyloid": "(eligible_core & ratio_ab42_ab40.notna()).astype(int)",
    "eligible_gfap_nfl": "(eligible_core & log_gfap.notna() & log_nfl.notna()).astype(int)",
    "eligible_gfap_amyloid":
        "(eligible_core & log_gfap.notna() & ratio_ab42_ab40.notna()).astype(int)",
    "eligible_nfl_amyloid":
        "(eligible_core & log_nfl.notna() & ratio_ab42_ab40.notna()).astype(int)",
    "eligible_full_blood":
        "(eligible_core & log_gfap.notna() & log_nfl.notna() & ratio_ab42_ab40.notna()).astype(int)",
}

manifest = {
    "manifest_schema_version": "2.0.0",
    "build": "Build 4 — MCI survival shared modeling dataset",
    "notebook_id": "03a_mci_survival_shared_modeling_dataset",
    "dataset": "mci_survival_shared_modeling_dataset",
    "version": VERSION,
    "purpose": ("shared analysis-ready survival-modeling dataset for feature-combination analysis, "
                "alternative survival-model comparison, centralized evaluation, and NACC "
                "harmonization; all 401 primary-cohort participants preserved; exactly 26 columns"),

    # ---- determinism contract ----
    "determinism": {
        "runtime_timestamps": "none — this manifest contains no build-time clock reading",
        "randomness_used": "none — this build requires no RNG",
        "byte_identical_across_clean_kernels": True,
        "self_hash_policy": ("a manifest cannot contain its own final SHA-256; it hashes every "
                             "other finalized artifact. Verify this file externally with shasum."),
    },

    # ---- provenance ----
    "git_commit": git("rev-parse", "HEAD"),
    "git_branch": git("rev-parse", "--abbrev-ref", "HEAD"),
    "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                    "numpy": np.__version__, "platform": platform.platform()},

    "input": {
        "path": f"outputs/01c_mci_survival_cohort_freeze/{INPUT_NAME}",
        "sha256": actual,
        "expected_sha256": EXPECTED_SHA,
        "checksum_verified": True,
        "n_participants": n, "n_events": n_ev, "n_censored": n_cn, "n_columns": ncol,
        "frozen_v1_cohort_unchanged": True,
        "cohort_redefinition": ("none — cohort, anchor, event, censoring and follow-up are taken "
                                "exactly as frozen; no raw ADNI table was re-joined"),
    },

    "output_dataset": {
        "path": f"outputs/03a_mci_survival_shared_modeling/{ds_path.name}",
        "n_rows": int(out.shape[0]), "n_columns": int(out.shape[1]),
        "n_events": int((out["event_indicator"] == 1).sum()),
        "n_censored": int((out["event_indicator"] == 0).sum()),
        "unique_participants": int(out["RID"].nunique()),
        "sort_order": "RID ascending",
        "format": "tab-separated, no dataframe index, NA for missing",
    },

    "final_columns": FINAL_COLUMNS,
    "source_to_output_column_mapping": SOURCE_TO_OUTPUT,
    "derived_column_formulas": DERIVED_FORMULAS,
    "natural_log_implementation": ("numpy.log — natural logarithm (base e). Every observed input "
                                   "to a log is finite and strictly positive (asserted); missing "
                                   "inputs remain missing in the derived output."),
    "amyloid_ratio_definition": {
        "formula": "ratio_ab42_ab40 = abeta42 / abeta40",
        "verification": ("recomputed from the carried components and compared against the frozen "
                         "column at rtol=1e-10, atol=1e-12"),
        "n_discrepancies": 0,
        "missing_rule": "NA where either component is NA (2 participants)",
    },

    "eligibility_definitions": {
        "eligible_core": "entry_age.notna() & APOE4_COUNT.notna() & log_ptau217.notna()",
        "eligible_gfap": "eligible_core & log_gfap.notna()",
        "eligible_nfl": "eligible_core & log_nfl.notna()",
        "eligible_amyloid": "eligible_core & ratio_ab42_ab40.notna()",
        "eligible_gfap_nfl": "eligible_core & log_gfap.notna() & log_nfl.notna()",
        "eligible_gfap_amyloid": "eligible_core & log_gfap.notna() & ratio_ab42_ab40.notna()",
        "eligible_nfl_amyloid": "eligible_core & log_nfl.notna() & ratio_ab42_ab40.notna()",
        "eligible_full_blood": ("eligible_core & log_gfap.notna() & log_nfl.notna() "
                                "& ratio_ab42_ab40.notna()"),
    },
    "eligibility_counts": {
        r["eligibility_flag"]: {"n_participants": int(r["n_participants"]),
                                "n_events": int(r["n_events"]),
                                "n_censored": int(r["n_censored"]),
                                "participant_fraction": float(r["participant_fraction"]),
                                "event_fraction": float(r["event_fraction"])}
        for _, r in counts.iterrows()
    },

    "missingness_structure": {
        "gfap_nfl_missing_n": int(out["gfap"].isna().sum()),
        "gfap_nfl_masks_identical": bool((out["gfap"].isna() == out["nfl"].isna()).all()),
        "amyloid_missing_n": int(out["ratio_ab42_ab40"].isna().sum()),
        "amyloid_missing_rids": amyloid_missing_rids,
        "amyloid_missing_all_events": True,
        "amyloid_and_gfap_missing_overlap": int(
            (out["ratio_ab42_ab40"].isna() & out["gfap"].isna()).sum()),
    },

    "prior_dementia_qc": {
        "anchor_cohort_A_participants_flagged": 8,
        "primary_cohort_C_participants_flagged": int(qc_flag.sum()),
        "primary_cohort_C_flagged_rids": qc_rids,
        "primary_cohort_C_diagnosis_records": qc_records,
        "events_among_flagged": qc_events,
        "record_vs_participant": ("the previously circulated '8' counts PARTICIPANTS in anchor "
                                  "cohort A, not diagnosis records. Four of those eight remain in "
                                  "the frozen primary cohort and carry 12 pre-anchor dementia "
                                  "diagnosis records between them (5+4+2+1)."),
        "pre_anchor_dementia_n": "participant-level field whose value is a diagnosis-RECORD count",
        "qc_dementia_on_or_before_anchor": "participant-level binary flag (n > 0)",
        "usage": ("QC and sensitivity analysis only — never a model predictor and never a filter "
                  "for the shared primary dataset"),
        "participants_excluded": 0,
    },

    "policy": {
        "participants_preserved": 401,
        "imputation": "none",
        "scaling_or_standardization": "none",
        "winsorization": "none",
        "feature_selection": "none",
        "model_fitting": "none",
        "cognitive_or_functional_variables_added": False,
        "cross_validation_folds_generated": False,
        "reduced_to_complete_case": False,
        "raw_biomarkers_carried_byte_identical": True,
        "prior_dementia_participants_retained": True,
        "frozen_v1_cohort_unchanged": True,
        "builds_1_to_3_artifacts_modified": False,
    },
    "reserved_downstream_cv_seed": 20260722,
    "reserved_downstream_cv_seed_note": ("reserved for DOWNSTREAM cross-validation only; this "
                                         "build generates no folds and consumes no randomness"),

    "modeling_protocol": {
        "M0": "entry_age + APOE4_COUNT",
        "M1": "entry_age + APOE4_COUNT + log_ptau217",
        "candidate_extension_features": ["log_gfap", "log_nfl", "ratio_ab42_ab40"],
        "incremental_comparison_rule": ("refit M1 on exactly the same participants as the extended "
                                        "model for every single-feature comparison"),
        "comparison_samples": {
            "gfap": {"n": 361, "events": 85},
            "nfl": {"n": 361, "events": 85},
            "amyloid": {"n": 399, "events": 83},
        },
        "head_to_head_common_subset": {"flag": "eligible_full_blood", "n": 359, "events": 83,
                                       "censored": 276},
        "refit_after_selection": ("a preferred feature set may later be refitted on the largest "
                                  "valid sample for its exact required features"),
        "prohibited": [
            "raw and log form of the same biomarker in one model",
            "APOE4_COUNT and APOE4_CARRIER in one model",
            "RID, anchor_date, QC fields, or eligible_* flags as predictors",
        ],
        "fitted_in_this_build": False,
    },

    "column_roles": {r["output_column"]: r["role"] for r in DD},
    "analysis_set_counts": counts.to_dict(orient="records"),
    "finalized_artifact_hashes": [
        {"path": rel, "sha256": sha256(p),
         **({"n_rows": int(shape[0]), "n_cols": int(shape[1])} if shape else {})}
        for rel, p, shape in FINALIZED
    ],
    "report": REPORT_REL,
    "notebook": "notebooks/03a_mci_survival_shared_modeling_dataset.ipynb",
    "protected_frozen_artifacts_unchanged": {f: HASHES_BEFORE[f] for f in PROTECTED},
}

MPATH = OUT_DIR / f"mci_survival_shared_modeling_manifest_{VERSION}.json"
MPATH.write_text(json.dumps(manifest, indent=2, default=str) + "\n")
print(f"  saved -> {MPATH.relative_to(PROJECT_ROOT)}")

# ---- manifest completeness assertions ----
check("manifest contains NO runtime timestamp field",
      not any("created" in k or "timestamp" in k or "_utc" in k for k in manifest),
      f"keys={[k for k in manifest if 'created' in k or 'timestamp' in k or '_utc' in k]}")
check("manifest records the current HEAD commit (no volatile dirty-state field)",
      isinstance(manifest["git_commit"], str) and len(manifest["git_commit"]) == 40
      and "dirty" not in json.dumps(manifest),
      manifest["git_commit"])
check("manifest records reserved_downstream_cv_seed == 20260722",
      manifest["reserved_downstream_cv_seed"] == 20260722)
check("manifest maps all 26 output columns back to a source column",
      set(manifest["source_to_output_column_mapping"]) == set(FINAL_COLUMNS))
check("manifest records the exact formula for every derived column",
      set(manifest["derived_column_formulas"]) ==
      {"APOE4_CARRIER", "log_ptau217", "log_gfap", "log_nfl", "ratio_ab42_ab40", *ELIG})
check("manifest hashes the dataset, dictionary, counts table AND the report (4 artifacts)",
      len(manifest["finalized_artifact_hashes"]) == 4
      and all(len(a["sha256"]) == 64 for a in manifest["finalized_artifact_hashes"])
      and any(a["path"] == REPORT_REL for a in manifest["finalized_artifact_hashes"]))
check("manifest does NOT attempt to contain its own SHA-256",
      sha256(MPATH) not in MPATH.read_text())
check("manifest records counts for every eligibility flag",
      set(manifest["eligibility_counts"]) == set(ELIG))
check("manifest states numpy.log, the ratio definition, and every no-* policy",
      "numpy.log" in manifest["natural_log_implementation"]
      and manifest["amyloid_ratio_definition"]["formula"] == "ratio_ab42_ab40 = abeta42 / abeta40"
      and manifest["policy"]["imputation"] == "none"
      and manifest["policy"]["scaling_or_standardization"] == "none"
      and manifest["policy"]["model_fitting"] == "none"
      and manifest["policy"]["cognitive_or_functional_variables_added"] is False
      and manifest["policy"]["cross_validation_folds_generated"] is False
      and manifest["policy"]["frozen_v1_cohort_unchanged"] is True)
check("manifest retains software/package versions",
      all(manifest["environment"][k] for k in ["python", "pandas", "numpy", "platform"]))

# ---- read-only guarantee ----
HASHES_AFTER = {f: sha256(FREEZE_DIR / f) for f in PROTECTED}
drift = [f for f in PROTECTED if HASHES_BEFORE[f] != HASHES_AFTER[f]]
check("ALL existing frozen v1 artifacts BYTE-IDENTICAL after this build",
      not drift, f"drifted: {drift}" if drift else f"{len(PROTECTED)} unchanged")
check("all outputs written under outputs/03a_mci_survival_shared_modeling/",
      all((OUT_DIR / x).exists() for x in [ds_path.name, dd_path.name, ct_path.name, MPATH.name]))

print("\nFinalized artifact hashes (manifest excludes its own):")
for a in manifest["finalized_artifact_hashes"]:
    print(f"  {a['path']:72s} {a['sha256'][:16]}...")
print(f"  {'(manifest itself, hashed externally)':72s} {sha256(MPATH)[:16]}...")

## 7. Summary

In [ ]:
print("=" * 96)
print("BUILD 4 — SHARED SURVIVAL-MODELING DATASET (03a, v1)")
print("=" * 96)
print(f"Assertions : {sum(a['passed'] for a in ASSERTIONS)} passed / "
      f"{sum(not a['passed'] for a in ASSERTIONS)} failed  (of {len(ASSERTIONS)})")
print(f"Input      : {INPUT_NAME}  sha256 {actual[:16]}...  VERIFIED")
print(f"Dataset    : {out.shape[0]} participants x {out.shape[1]} columns "
      f"({int((out['event_indicator'] == 1).sum())} events, "
      f"{int((out['event_indicator'] == 0).sum())} censored)")
print("             ALL 401 preserved; optional biomarkers NA where missing; no imputation")
print()
print("Analysis sets (per-participant membership; the shared file is never pre-filtered):")
print(f"  {'flag':24s} {'N':>4s} {'ev':>4s} {'cens':>5s}  {'frac_N':>18s} {'frac_ev':>18s}")
for _, r in counts.iterrows():
    print(f"  {r['analysis_set']:24s} {int(r['n_participants']):>4d} {int(r['n_events']):>4d} "
          f"{int(r['n_censored']):>5d}  {r['participant_fraction']:>18.16f} "
          f"{r['event_fraction']:>18.16f}")
print()
print(f"Missingness structure: GFAP/NfL missing for the same {int(gfap_missing.sum())} "
      f"participants (identical masks); amyloid missing for {int(amyloid_missing.sum())} "
      f"(RIDs {amyloid_missing_rids}, both events);")
print(f"                      the two groups are disjoint -> full_blood = 401 - 40 - 2 = "
      f"{int(out['eligible_full_blood'].sum())}")
print()
print(f"Prior dementia (QC only, never a predictor, never a filter):")
print(f"  8 PARTICIPANTS flagged in anchor cohort A -> {int(qc_flag.sum())} remain in the frozen "
      f"primary cohort (RIDs {qc_rids}),")
print(f"  carrying {qc_records} diagnosis RECORDS between them ("
      f"{'+'.join(str(int(v)) for v in out.loc[qc_flag, 'pre_anchor_dementia_n'])}); "
      f"{qc_events} of the 4 is an event. All 4 RETAINED; adjudication pending.")
print()
print("Downstream protocol documented (NOT fitted here): M0 = entry_age + APOE4_COUNT; "
      "M1 = M0 + log_ptau217;")
print("  extensions from {log_gfap, log_nfl, ratio_ab42_ab40}; refit M1 on the identical "
      "participant set for each")
print("  incremental comparison; head-to-head on eligible_full_blood (359 / 83 / 276). "
      f"Reserved downstream CV seed: {manifest['reserved_downstream_cv_seed']}.")
print()
print("Load: outputs/03a_mci_survival_shared_modeling/mci_survival_shared_modeling_dataset_v1.tsv")
print("Frozen v1 unchanged. No imputation, scaling, feature selection, models, cognition, CV folds.")
print("Deterministic: no runtime timestamps or randomness in any generated artifact.")
print("=" * 96)